# Logistic Regression Training

Train and evaluate the Logistic Regression pipelines using the cleaned dataset exported by `01-preprocessing.ipynb`.

## Data Loading

Load the existing cleaned CSV. The two paths support running from either the repository root or this notebook's folder.


In [ ]:
from pathlib import Path
import pandas as pd

cleaned_dataset_path = Path("diadb-training/cleaned_diabetes.csv")
if not cleaned_dataset_path.is_file():
    cleaned_dataset_path = Path("cleaned_diabetes.csv")

df = pd.read_csv(cleaned_dataset_path)


## Model Input Setup

Use the same target, numeric features, and categorical feature as the original Logistic Regression workflow.


In [ ]:
import matplotlib.pyplot as plt

target_col = "diabetic"
numerical_cols = ["pulse_rate", "systolic_bp", "diastolic_bp", "glucose", "bmi"]


# Model Development and Evaluation

## Train-Test Split

Separate predictors from the target, encode the target labels, and create stratified training and test sets. Use the same `random_state` and split settings in other model notebooks for comparable results.


In [ ]:
X = df.drop(target_col, axis=1)

y = df[target_col].map({
    "No": 0,
    "Yes": 1
})


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


## Feature Preparation

Specify the categorical predictor that will be one-hot encoded by the preprocessing pipeline.


In [ ]:
categorical_cols = ["gender"]


## Training Pipeline Setup

Install and import the pipeline components, then define cross-validation, preprocessing, and model pipelines.

In [ ]:
%pip install imbalanced-learn

### Cross-Validation Strategy

Configure stratified folds for model selection.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### Feature Encoding and Scaling

Build the column-wise preprocessing step using the existing one-hot encoding and standard scaling configuration.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_cols
        )
    ]
)

### Logistic Regression Pipelines

Set up the baseline classifier and compare pipeline variants with and without SMOTE.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_regression = LogisticRegression(random_state=42, max_iter=1000)

In [ ]:
from imblearn.pipeline import Pipeline

log_reg_without_smote = Pipeline(
    [("preprocessor", preprocessor), ("classifier", logistic_regression)]
)

In [ ]:
from imblearn.over_sampling import SMOTE

log_reg_with_smote = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", logistic_regression),
    ]
)

## Hyperparameter Tuning

Define the existing search space and tune both pipeline variants using the training data.

In [ ]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__solver": ["liblinear", "lbfgs"],
    "classifier__max_iter": [20, 40, 50,100]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_without_smote = GridSearchCV(
    estimator=log_reg_without_smote,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

In [ ]:
grid_without_smote.fit(X_train, y_train)

In [ ]:
print(grid_without_smote.best_params_)

In [ ]:
print(grid_without_smote.best_score_)

In [ ]:
best_without_smote = grid_without_smote.best_estimator_

In [ ]:
grid_with_smote = GridSearchCV(
    estimator=log_reg_with_smote,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

In [ ]:
grid_with_smote.fit(X_train, y_train)

In [ ]:
print(grid_with_smote.best_params_)
print(grid_with_smote.best_score_)

In [ ]:
best_with_smote = grid_with_smote.best_estimator_

## Nested Cross-Validation: Training vs. Validation Loss and F1-Score

Estimate generalization on the training split with nested cross-validation. Each outer fold holds out data for evaluation, while `GridSearchCV` tunes hyperparameters using only that outer fold's training data. The inner folds select hyperparameters; the outer folds estimate performance. Training scores are diagnostic, while the outer validation scores provide the less biased comparison.

In [ ]:
cv_models = {
    "Without SMOTE": grid_without_smote,
    "With SMOTE": grid_with_smote,
}

In [ ]:
from sklearn.model_selection import cross_validate

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2025)

cv_comparison = {}
for model_name, model in cv_models.items():
    cv_comparison[model_name] = cross_validate(
        model,
        X_train,
        y_train,
        cv=outer_cv,
        scoring={"f1": "f1", "neg_log_loss": "neg_log_loss"},
        return_train_score=True,
        n_jobs=1,
    )

In [ ]:
import numpy as np

folds = np.arange(1, outer_cv.get_n_splits(X_train, y_train) + 1)
colors = {"Without SMOTE": "tab:blue", "With SMOTE": "tab:orange"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for model_name, scores in cv_comparison.items():
    color = colors[model_name]
    train_loss = -scores["train_neg_log_loss"]
    validation_loss = -scores["test_neg_log_loss"]

    axes[0].plot(
        folds, train_loss, marker="o", color=color, label=f"{model_name} - Training"
    )
    axes[0].plot(
        folds,
        validation_loss,
        marker="o",
        linestyle="--",
        color=color,
        label=f"{model_name} - Validation",
    )
    axes[1].plot(folds, scores["test_f1"], marker="o", color=color, label=model_name)

    print(
        f"{model_name} - Nested CV F1: "
        f"{scores['test_f1'].mean():.4f} +/- {scores['test_f1'].std():.4f}"
    )

axes[0].set_title("Nested CV: Training vs. Outer Validation Log Loss")
axes[0].set_xlabel("Outer cross-validation fold")
axes[0].set_ylabel("Log loss (lower is better)")
axes[0].set_xticks(folds)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title("Outer Validation F1-Score by Fold")
axes[1].set_xlabel("Outer cross-validation fold")
axes[1].set_ylabel("F1-score (higher is better)")
axes[1].set_xticks(folds)
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Test-Set Evaluation

Calculate the existing classification metrics for the selected models and visualize their confusion matrices.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def test_performance(model, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{model_name}")

    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"F1:        {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
    print(f"PR-AUC:    {average_precision_score(y_test, y_prob):.4f}")

In [ ]:
test_performance(
    best_without_smote,
    "Logistic Regression - Without SMOTE"
)

test_performance(
    best_with_smote,
    "Logistic Regression - With SMOTE"
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for model, name in [
    (best_without_smote, "Without SMOTE"),
    (best_with_smote, "With SMOTE"),
]:
    y_pred = model.predict(X_test)

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["No Diabetes", "Diabetes"]
    )

    plt.title(name)
    plt.show()